# Module 5 — Inverse Kinematics
## Unit 5 — Numerical Inverse Kinematics in Practice
### Lesson 5.2 — The Jacobian-Transpose and Damped Least Squares

*Physical AI Curriculum · hands-on notebook. Run **Kernel → Restart & Run All**.*


## Demonstration (§7)
Near a singular pose the pseudoinverse step explodes; DLS stays bounded.


In [ ]:
import numpy as np
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

def fk_two_link(t1, t2, L1, L2):
    return np.array([L1*np.cos(t1)+L2*np.cos(t1+t2), L1*np.sin(t1)+L2*np.sin(t1+t2)])

def jacobian_2link(t1, t2, L1, L2):
    s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12, -L2*s12],[L1*c1+L2*c12, L2*c12]])

def ik_2link_closed(x, y, L1, L2):
    c2=(x*x+y*y-L1*L1-L2*L2)/(2*L1*L2)
    if c2<-1-1e-9 or c2>1+1e-9: return []
    c2=max(-1.0,min(1.0,c2)); sols=[]
    for sign in (+1.0,-1.0):
        s2=sign*np.sqrt(max(0.0,1-c2*c2)); t2=np.arctan2(s2,c2)
        t1=np.arctan2(y,x)-np.arctan2(L2*np.sin(t2),L1+L2*np.cos(t2))
        sols.append((t1,t2))
        if abs(s2)<1e-6: break
    return sols

def step_pinv(J,e): return np.linalg.pinv(J)@e
def step_transpose(J,e,alpha=0.3): return alpha*J.T@e
def step_dls(J,e,lam=0.1): return J.T@np.linalg.inv(J@J.T+lam*lam*np.eye(len(e)))@e

L1,L2=0.4,0.3
# near-singular: theta2 ~ 2 degrees (almost straight)
J=jacobian_2link(np.radians(20),np.radians(2),L1,L2)
e=np.array([0.02,0.0])
print("|pinv step|     =",round(np.linalg.norm(step_pinv(J,e)),3))
print("|transpose step|=",round(np.linalg.norm(step_transpose(J,e)),3))
print("|dls step|      =",round(np.linalg.norm(step_dls(J,e)),3))


## Coding Exercise (§8)
Show pinv magnitude >> dls magnitude near singularity; ik_dls converges.


In [ ]:
# YOUR CODE HERE
J=jacobian_2link(np.radians(20),np.radians(2),L1,L2); e=np.array([0.02,0.0])
assert np.linalg.norm(step_pinv(J,e)) > 5*np.linalg.norm(step_dls(J,e))
def ik_dls(target,theta0,L1,L2,lam=0.05,tol=1e-5,max_iter=200):
    theta=np.array(theta0,float); target=np.array(target,float)
    for it in range(max_iter):
        err=target-fk_two_link(theta[0],theta[1],L1,L2)
        if np.linalg.norm(err)<tol: return theta,it
        J=jacobian_2link(theta[0],theta[1],L1,L2)
        theta=theta+step_dls(J,err,lam)
    return theta,max_iter
sol,iters=ik_dls([0.5,0.2],np.radians([10,20]),L1,L2)
assert np.allclose(fk_two_link(sol[0],sol[1],L1,L2),[0.5,0.2],atol=1e-4)
print("DLS bounded + converges OK")


## Check your work


In [ ]:
# transpose step is downhill: it reduces ||e||^2
theta=np.radians([30,40]); target=np.array([0.5,0.2])
err=target-fk_two_link(theta[0],theta[1],L1,L2)
J=jacobian_2link(theta[0],theta[1],L1,L2)
new=theta+step_transpose(J,err,0.3)
err2=target-fk_two_link(new[0],new[1],L1,L2)
assert np.linalg.norm(err2)<np.linalg.norm(err)
print("All checks passed.")
